# LongFlow — Gate Night 5 (where does the poison enter?)

Runtime: **L4 GPU**. ~1.5 h, ~$2–3. Five independent arms; every artifact
mirrors to Drive per run; reruns skip completed work. Pre-registered criteria:
`experiments/p1_flow_head/NOTES.md` (Gate Night 5 entry).

| cell | arm | decides |
|---|---|---|
| 4 | **F: feedback-path ablation** — euler4 closed loop, acoustic feedback intact / zeroed / running-mean / noised | is the fix plumbing or training? (dots.tts hypothesis) |
| 6 | **D: FD metric backfill** — σ-VAE re-encode GN4+teacher wavs, per-window Fréchet distance | the publishable dose-response figure (std is not a collapse metric) |
| 8 | **R: reseed floor n≤40** — teacher, 2 seeds per held-out utt | the noise band every WER comparison needs (blocking item 1) |
| 10 | **S: chunked-parallel seam test** — 2-speaker script, parallel chunks, crossfade | is the product path's stitching audible? (JOSH LISTENS) |
| 12 | **H: batch hygiene** — batch=[A,A] identical pair | cross-contamination test that bit-determinism arguments actually need |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
import numpy as np
import soundfile as sf

if not os.path.exists("/content/LongFlow/src"):
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone LongFlow or drag bundle"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.flow_head.cfm import euler_sample
from src.flow_head.integration import FlowHeadPatch
from src.flow_head.trainer import load_checkpoint

CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
GATE4_DIR = "/content/drive/MyDrive/longflow_gate4"
OUT = "/content/gate_night5"
DRIVE_OUT = "/content/drive/MyDrive/longflow_gate5"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

head20, mean20, std20 = load_checkpoint(f"{CKPT_DIR}/full10k_20k.pt")
head20 = head20.to("cuda")

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def save_wav(tag, wav, extra=None):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    row = {"tag": tag, "audio_s": round(len(wav)/24000, 1)}
    if extra: row.update(extra)
    report["runs"].append(row)
    json.dump(report, open(f"{DRIVE_OUT}/gate_night5_report.json", "w"), indent=2)
    print(row, flush=True)

def done(tag):
    if os.path.exists(f"{DRIVE_OUT}/{tag}.wav"):
        print(f"skip {tag}", flush=True)
        return True
    return False

report = {"runs": []}
print("READY")


In [ ]:
# shared inputs
sents = []
for f in sorted(glob.glob(f"{TRAIN_CACHE_DIR}/*.pt"))[-300:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
prompts = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*_prompt.wav"))
P0, P1 = prompts[0], prompts[1] if len(prompts) > 1 else prompts[0]

def turnscript(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return "\n".join(turns) + "\n"

# ~5-min turn-split script for the ablation arms (~800 words)
ABL_WORDS = []
w = 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800: break
ABL_SCRIPT = turnscript(ABL_WORDS)
print("ablation script:", w, "words")
report["ablation_words"] = w


## 4. Arm F — feedback-path ablation ("where does the poison enter?")

GN4 established the loop amplifies any head imperfection through the acoustic
feedback channel — the model re-reading its own generated latents. dots.tts
avoids this architecturally (LM sees semantic summaries only). Here we cut or
corrupt the acoustic connector's OUTPUT at inference and measure survival:

- **base** — untouched (GN4 control, re-rendered at 5-min length)
- **zero** — acoustic connector output zeroed (LM gets a null acoustic embed)
- **mean** — output replaced by its own running mean (in-distribution null;
  the fairer control — pure zeros are themselves OOD)
- **noise** — output + N(0, σ=0.5·running std) (does corruption ≈ our head's
  imperfection reproduce collapse with the TEACHER-quality signal absent?)

All with the 20K head @ euler4 (fastest collapse → most sensitive assay),
5-min script, seed 0. If zero/mean survive ≫ base, the poison is confirmed
acoustic-channel and an inference-time fix exists. ~25 min.


In [ ]:
class ConnectorIntervention:
    def __init__(self, module, mode):
        self.module, self.mode = module, mode
        self.n, self.mu, self.var = 0, None, None
        self.h = None
    def __enter__(self):
        if self.mode == "base":
            return self
        def hook(mod, args, out):
            t = out[0] if isinstance(out, tuple) else out
            with torch.no_grad():
                if self.mu is None:
                    self.mu = t.mean().detach(); self.var = t.var().detach()
                else:
                    self.mu = 0.99*self.mu + 0.01*t.mean().detach()
                    self.var = 0.99*self.var + 0.01*t.var().detach()
                if self.mode == "zero":
                    new = torch.zeros_like(t)
                elif self.mode == "mean":
                    new = torch.full_like(t, self.mu.item())
                elif self.mode == "noise":
                    new = t + torch.randn_like(t) * (0.5 * self.var.sqrt())
            return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new
        self.h = self.module.register_forward_hook(hook)
        return self
    def __exit__(self, *exc):
        if self.h: self.h.remove()

for mode in ("base", "zero", "mean", "noise"):
    tag = f"f_abl_{mode}"
    if done(tag):
        continue
    torch.manual_seed(0)
    with ConnectorIntervention(model.model.acoustic_connector, mode), \
         FlowHeadPatch(model, head20, mean20, std20, nfe=4, sway=0.0,
                       sampler=euler_sample) as patch, torch.inference_mode():
        out = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=3000)
    wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    zs = torch.cat(patch.latents) if patch.latents else torch.zeros(1)
    save_wav(tag, wav, {"latent_std": round(float(zs.std()), 3),
                        "frames": patch.calls})


## 6. Arm D — Fréchet-distance backfill (the honest collapse metric)

Scalar latent std failed twice as a collapse metric (E3, GN4 euler4). Here:
re-encode the GN4 renders + the GN3 teacher render through the σ-VAE encoder,
compute per-10s-window Fréchet distance (mean+cov Gaussian) against the
teacher's window distribution. Output: FD-vs-time curves for
euler4 / heun8 / teacher-self → the publishable dose-response figure.
Also runs on Arm F renders if present. ~10 min.


In [ ]:
def encode_latents(path):
    x, sr = sf.read(path, dtype="float32")
    z_all = []
    step = 24000 * 30
    with torch.inference_mode():
        for i in range(0, len(x), step):
            seg = torch.from_numpy(x[i:i+step])[None, None].to("cuda", torch.bfloat16)
            enc = model.model.acoustic_tokenizer.encode(seg)
            z = enc[0] if isinstance(enc, tuple) else enc
            if hasattr(z, "sample"):
                z = z.sample() if callable(z.sample) else z.sample
            z_all.append(z.squeeze(0).float().cpu())
    return torch.cat(z_all)  # [T, d]

def fd_curve(z, ref_mu, ref_cov, win=75):  # 75 frames = 10 s @ 7.5 Hz
    import scipy.linalg
    out = []
    for i in range(0, len(z) - win, win):
        w = z[i:i+win].numpy()
        mu, cov = w.mean(0), np.cov(w.T)
        d = mu - ref_mu
        covmean = scipy.linalg.sqrtm(cov @ ref_cov)
        if np.iscomplexobj(covmean): covmean = covmean.real
        out.append(float(d @ d + np.trace(cov + ref_cov - 2*covmean)))
    return out

teacher_z = encode_latents(f"{GATE4_DIR}/../longflow_gate3/t1_turnsplit_p0.wav") \
    if os.path.exists(f"{GATE4_DIR}/../longflow_gate3/t1_turnsplit_p0.wav") else None
assert teacher_z is not None, "teacher render not on Drive"
half = len(teacher_z) // 2
ref_mu, ref_cov = teacher_z[:half].numpy().mean(0), np.cov(teacher_z[:half].numpy().T)

fd = {"teacher_self": fd_curve(teacher_z[half:], ref_mu, ref_cov)}
for tag in ("a_head20_turnsplit_euler4", "a_head20_turnsplit_heun8"):
    p = f"{GATE4_DIR}/{tag}.wav"
    if os.path.exists(p):
        fd[tag] = fd_curve(encode_latents(p), ref_mu, ref_cov)
for mode in ("base", "zero", "mean", "noise"):
    p = f"{DRIVE_OUT}/f_abl_{mode}.wav"
    if os.path.exists(p):
        fd[f"f_abl_{mode}"] = fd_curve(encode_latents(p), ref_mu, ref_cov)
report["fd_curves"] = fd
json.dump(report, open(f"{DRIVE_OUT}/gate_night5_report.json", "w"), indent=2)
for k, v in fd.items():
    print(f"{k}: first={v[0]:.1f} med={sorted(v)[len(v)//2]:.1f} last={v[-1]:.1f} n={len(v)}")


## 8. Arm R — reseed floor at n≤40 (blocking item 1, finally)

GN1's n=6 floor was bimodal and unusable (median 0.000, mean carried by one
proper-noun utterance). Teacher renders each held-out utterance twice
(seeds 0/1); Mac computes median + IQR pair-WER. ~15 min.


In [ ]:
eval_files = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*.pt"))[:40]
manifest = {}
for f in eval_files:
    d = torch.load(f, weights_only=True)
    uid = d.get("utt_id", os.path.basename(f)[:-3])
    text = d["text"].strip()
    pw = f"{EVAL_CACHE_DIR}/{uid}_prompt.wav"
    prompt = pw if os.path.exists(pw) else P0
    manifest[uid] = text
    for seed in (0, 1):
        tag = f"r_{uid}_s{seed}"
        if done(tag):
            continue
        torch.manual_seed(seed)
        with torch.inference_mode():
            out = model.generate(**gen_inputs([f"Speaker 1: {text}\n"], [[prompt]]),
                                 tokenizer=processor.tokenizer,
                                 cfg_scale=1.3, max_new_tokens=800)
        save_wav(tag, out.speech_outputs[0].detach().float().cpu().numpy().squeeze())
report["reseed_manifest"] = manifest
json.dump(report, open(f"{DRIVE_OUT}/gate_night5_report.json", "w"), indent=2)
print(len(manifest), "utterances x 2 seeds")


## 10. Arm S — chunked-parallel seam test (the product path, heard)

2-speaker dialogue (~750 words) built from the pool, split into 4 chunks at
turn boundaries, all 4 rendered as ONE batch with the same two voice prompts,
concatenated with a 0.25 s crossfade. TEACHER head (product config).
**JOSH LISTENS to the 3 seams.** ~5 min.


In [ ]:
dlg, w, sp = [], 0, 1
for s in pool[100:]:
    dlg.append(f"Speaker {sp}: {s}")
    w += len(s.split()); sp = 2 if sp == 1 else 1
    if w >= 750: break
lines_per = (len(dlg) + 3) // 4
chunks = ["\n".join(dlg[i*lines_per:(i+1)*lines_per]) + "\n" for i in range(4)]
chunks = [c for c in chunks if c.strip()]
print(len(chunks), "chunks;", [len(c.split()) for c in chunks])

if not done("s_chunked_stitched"):
    torch.manual_seed(0)
    with torch.inference_mode():
        out = model.generate(**gen_inputs(chunks, [[P0, P1]]*len(chunks)),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=2500)
    pieces = [o.detach().float().cpu().numpy().squeeze() for o in out.speech_outputs]
    for i, p in enumerate(pieces):
        save_wav(f"s_chunk_{i}", p)
    xf = int(0.25 * 24000)
    stitched = pieces[0]
    for p in pieces[1:]:
        fade = np.linspace(1, 0, xf)
        stitched = np.concatenate([
            stitched[:-xf],
            stitched[-xf:]*fade + p[:xf]*(1-fade),
            p[xf:]])
    save_wav("s_chunked_stitched", stitched)


## 12. Arm H — batch hygiene: the [A, A] test

Two IDENTICAL rows in one batch, same seed. Within one batched kernel call the
two rows share every input — if padding/masking is clean, the outputs must be
IDENTICAL to each other (this, unlike solo-vs-batch bit-equality, is a fair
determinism criterion). Any divergence = genuine cross-contamination. ~2 min.


In [ ]:
text = f"Speaker 1: {pool[0]}\n"
torch.manual_seed(0)
with torch.inference_mode():
    out = model.generate(**gen_inputs([text, text], [[P0], [P0]]),
                         tokenizer=processor.tokenizer,
                         cfg_scale=1.3, max_new_tokens=400)
a = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
b = out.speech_outputs[1].detach().float().cpu().numpy().squeeze()
same_len = len(a) == len(b)
maxdiff = float(np.abs(a[:min(len(a),len(b))] - b[:min(len(a),len(b))]).max())
report["hygiene_AA"] = {"same_length": bool(same_len),
                        "len_a": len(a), "len_b": len(b), "maxdiff": maxdiff}
json.dump(report, open(f"{DRIVE_OUT}/gate_night5_report.json", "w"), indent=2)
print(report["hygiene_AA"])
save_wav("h_AA_row0", a); save_wav("h_AA_row1", b)


## 13. Bundle

`gate_night5_bundle.zip` (zips from the Drive mirror). Mac:
`unzip -o ~/Downloads/gate_night5_bundle.zip -d experiments/p1_flow_head/audio/gate_night5`
then `.venv/bin/python experiments/p1_flow_head/score_gate_night5.py`.


In [ ]:
import zipfile
json.dump(report, open(f"{DRIVE_OUT}/gate_night5_report.json", "w"), indent=2)
with zipfile.ZipFile("/content/gate_night5_bundle.zip", "w") as z:
    for f in glob.glob(f"{DRIVE_OUT}/*"):
        z.write(f, os.path.basename(f))
print("download /content/gate_night5_bundle.zip")
